# 03 — Match Oscar Nominations to IMSDb Scripts

This notebook loads the Oscar nominations dataset and attempts to match each nominated film to an available script in the IMSDb index using fuzzy string matching.

**Run this notebook after** `01_scrape_scripts.ipynb`.

**Inputs:**
- `full_data.csv` — complete Oscar nominations dataset (97 ceremonies, ~12,014 records)
- IMSDb script index (fetched live from the web)

**Goal:** Identify which Oscar-nominated films have a corresponding script available for text analysis.

In [12]:
#%pip install pandas

## 1. Load Oscar Nominations Dataset

Load `full_data.csv` (tab-delimited). Contains all nominations across 97 Oscar ceremonies (1927/28–2024), with 16 columns including film title, nominee, category, and winner flag.

In [13]:
import pandas as pd

# Load the CSV file into a dataframe (tab-delimited)
df = pd.read_csv('full_data.csv', sep='\t')
df

,Ceremony,Year,Class,CanonicalCategory,Category,NomId,Film,FilmId,Name,Nominees,NomineeIds,Winner,Detail,Note,Citation,MultifilmNomination
0,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,an0051251,The Noose,tt0019217,Richard Barthelmess,Richard Barthelmess,nm0001932,NaN,Nickie Elkins,NaN,NaN,True
1,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,an0051252,The Patent Leather Kid,tt0018253,Richard Barthelmess,Richard Barthelmess,nm0001932,NaN,The Patent Leather Kid,NaN,NaN,True
2,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,an0051250a,The Last Command,tt0019071,Emil Jannings,Emil Jannings,nm0417837,True,General Dolgorucki [Grand Duke Sergius Alexander],NaN,NaN,True
3,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,an0051250b,The Way of All Flesh,tt0019553,Emil Jannings,Emil Jannings,nm0417837,True,August Schilling,NaN,NaN,True
4,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,an0051255,A Ship Comes In,tt0018389,Louise Dresser,Louise Dresser,nm0237571,NaN,Mrs. Pleznik,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12009,97,2024,SciTech,SCIENTIFIC AND TECHNICAL AWARD (Technical Achi...,SCIENTIFIC AND TECHNICAL AWARD (Technical Achi...,NaN,NaN,NaN,NaN,"Dustin Brooks, Colin Decker",NaN,True,NaN,Fire for Hire's gel first publicly demonstrate...,To DUSTIN BROOKS and COLIN DECKER for the deve...,NaN
12010,97,2024,SciTech,SCIENTIFIC AND TECHNICAL AWARD (Technical Achi...,SCIENTIFIC AND TECHNICAL AWARD (Technical Achi...,NaN,NaN,NaN,NaN,"Attila T. Áfra, Timo Aila",NaN,True,NaN,Open Image Denoise is an open-source library t...,To ATTILA T. ÁFRA for the creation of Intel Op...,NaN
12011,97,2024,SciTech,SCIENTIFIC AND TECHNICAL AWARD (Technical Achi...,SCIENTIFIC AND TECHNICAL AWARD (Technical Achi...,NaN,NaN,NaN,NaN,Mark Noel,NaN,True,NaN,The NACMO series of modular motion bases enabl...,To MARK NOEL for adapting and enhancing the sa...,NaN
12012,97,2024,SciTech,SCIENTIFIC AND TECHNICAL AWARD (Technical Achi...,SCIENTIFIC AND TECHNICAL AWARD (Technical Achi...,NaN,NaN,NaN,NaN,"Su Tie, Bei Shimen, Zhao Yanchong",NaN,True,NaN,Utilizing three-axis stabilization through mul...,To SU TIE for the development of the sensor an...,NaN


## 2. Explore and Filter Relevant Categories

Identify the award categories relevant to this analysis — acting (Actor/Actress), writing, and Best Picture — and filter the dataset down to those records.

In [14]:
# Get all unique values of the Category column
unique_categories = df['Category'].unique()

# Filter for categories related to writing, best movie, actor, and actress
keywords = ['WRITING', 'BEST PICTURE', 'ACTOR', 'ACTRESS']
filtered_categories = [cat for cat in unique_categories if any(keyword in cat.upper() for keyword in keywords)]

print("Filtered Categories:")
for cat in sorted(filtered_categories):
    print(f"  - {cat}")

Filtered Categories:
  - ACTOR
  - ACTOR IN A LEADING ROLE
  - ACTOR IN A SUPPORTING ROLE
  - ACTRESS
  - ACTRESS IN A LEADING ROLE
  - ACTRESS IN A SUPPORTING ROLE
  - BEST PICTURE
  - WRITING
  - WRITING (Adaptation)
  - WRITING (Adapted Screenplay)
  - WRITING (Motion Picture Story)
  - WRITING (Original Motion Picture Story)
  - WRITING (Original Screenplay)
  - WRITING (Original Story)
  - WRITING (Screenplay Adapted from Other Material)
  - WRITING (Screenplay Based on Material Previously Produced or Published)
  - WRITING (Screenplay Based on Material from Another Medium)
  - WRITING (Screenplay Written Directly for the Screen)
  - WRITING (Screenplay Written Directly for the Screen--based on factual material or on story material not previously published or produced)
  - WRITING (Screenplay)
  - WRITING (Screenplay--Adapted)
  - WRITING (Screenplay--Original)
  - WRITING (Screenplay--based on material from another medium)
  - WRITING (Story and Screenplay)
  - WRITING (Story and

In [15]:
# Filter the dataframe to include only the filtered categories
df_filtered = df[df['Category'].isin(filtered_categories)]

# Display the filtered dataframe
print(f"Original dataframe shape: {df.shape}")
print(f"Filtered dataframe shape: {df_filtered.shape}")
print("\nFiltered dataframe:")
df_filtered

Original dataframe shape: (12014, 16)
Filtered dataframe shape: (3268, 16)

Filtered dataframe:


,Ceremony,Year,Class,CanonicalCategory,Category,NomId,Film,FilmId,Name,Nominees,NomineeIds,Winner,Detail,Note,Citation,MultifilmNomination
0,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,an0051251,The Noose,tt0019217,Richard Barthelmess,Richard Barthelmess,nm0001932,NaN,Nickie Elkins,NaN,NaN,True
1,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,an0051252,The Patent Leather Kid,tt0018253,Richard Barthelmess,Richard Barthelmess,nm0001932,NaN,The Patent Leather Kid,NaN,NaN,True
2,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,an0051250a,The Last Command,tt0019071,Emil Jannings,Emil Jannings,nm0417837,True,General Dolgorucki [Grand Duke Sergius Alexander],NaN,NaN,True
3,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,an0051250b,The Way of All Flesh,tt0019553,Emil Jannings,Emil Jannings,nm0417837,True,August Schilling,NaN,NaN,True
4,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,an0051255,A Ship Comes In,tt0018389,Louise Dresser,Louise Dresser,nm0237571,NaN,Mrs. Pleznik,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11991,97,2024,Writing,WRITING (Original Screenplay),WRITING (Original Screenplay),fake_nomid035,Anora,tt28607951,Written by Sean Baker,Sean Baker,nm0048918,True,NaN,NaN,NaN,NaN
11992,97,2024,Writing,WRITING (Original Screenplay),WRITING (Original Screenplay),fake_nomid036,The Brutalist,tt8999762,"Written by Brady Corbet, Mona Fastvold","Brady Corbet, Mona Fastvold","nm1227232,nm2178754",NaN,NaN,NaN,NaN,NaN
11993,97,2024,Writing,WRITING (Original Screenplay),WRITING (Original Screenplay),fake_nomid037,A Real Pain,tt21823606,Written by Jesse Eisenberg,Jesse Eisenberg,nm0251986,NaN,NaN,NaN,NaN,NaN
11994,97,2024,Writing,WRITING (Original Screenplay),WRITING (Original Screenplay),fake_nomid039,September 5,tt28082769,"Written by Moritz Binder, Tim Fehlbaum; Co-Wri...","Moritz Binder, Tim Fehlbaum, Alex David","nm9415549,nm2959497,nm1017048",NaN,NaN,NaN,NaN,NaN


## 3. Extract Unique Nominated Films

Deduplicate the filtered records to get a list of film titles that will be looked up in the IMSDb index. Result: **1,674 unique films**.

In [16]:
# Get unique films from the filtered dataframe as a list
# Filter out NaN values and ensure all are strings
unique_films = [film for film in df_filtered['Film'].unique() if pd.notna(film) and isinstance(film, str)]
print(f"Total unique films: {len(unique_films)}")
print(f"Type: {type(unique_films)}")
print("\nUnique Films:")
for film in unique_films:
    print(f"  - {film}")

Total unique films: 1674
Type: <class 'list'>

Unique Films:
  - The Noose
  - The Patent Leather Kid
  - The Last Command
  - The Way of All Flesh
  - A Ship Comes In
  - 7th Heaven
  - Street Angel
  - Sunrise
  - Sadie Thompson
  - The Jazz Singer
  - Glorious Betsy
  - Underworld
  - The Private Life of Helen of Troy
  - Thunderbolt
  - In Old Arizona
  - Alibi
  - The Valiant
  - The Patriot
  - Madame X
  - The Barker
  - The Letter
  - The Divine Lady
  - The Broadway Melody
  - Coquette
  - The Cop
  - The Leatherneck
  - Sal of Singapore
  - Skyscraper
  - The Last of Mrs. Cheyney
  - Our Dancing Daughters
  - A Woman of Affairs
  - Wonder of Women
  - Disraeli
  - The Green Goddess
  - The Big House
  - The Big Pond
  - The Love Parade
  - Bulldog Drummond
  - Condemned
  - The Rogue Song
  - The Devil's Holiday
  - Sarah and Son
  - Anna Christie
  - Romance
  - The Divorcee
  - Their Own Desire
  - The Trespasser
  - All Quiet on the Western Front
  - Street of Chance
  - A

## 4. Fetch the IMSDb Script Index

Scrape the current list of available scripts from IMSDb. This builds the target set for fuzzy matching.

In [17]:
#%pip install requests beautifulsoup4
#%pip install thefuzz

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from thefuzz import process, fuzz

def fetch_imsdb_index():
    url = "https://imsdb.com/all-scripts.html"
    headers = {'User-Agent': 'Mozilla/5.0'}
    res = requests.get(url, headers=headers)
    soup = BeautifulSoup(res.text, 'html.parser')
    
    dict_urls = {}
    # In IMSDb, movie links are inside table cells or paragraphs
    for link in soup.find_all('a', href=True):
        if "/Movie Scripts/" in link['href']:
            # Clean the title text: remove newlines and extra whitespace
            title = " ".join(link.text.split())
            
            # Convert to plain text URL and clean spaces in the URL
            dirty_path = link['href'].replace("/Movie Scripts/", "/scripts/").replace(" Script.html", ".html")
            final_url = "https://imsdb.com" + dirty_path.replace(" ", "%20")
            
            dict_urls[title] = final_url
            
    return dict_urls

# Fetch the index
online_index = fetch_imsdb_index()


## 5a. Matching Attempt 1 — Token Set Ratio

Uses `fuzz.token_set_ratio`, which is word-order agnostic (treats "The Big House" and "Big" as equivalent). This gives a **27.72% match rate** but produces many false positives (e.g., "The Big House" matching "Big", "Romance" matching "True Romance").

In [ ]:
final_mapping = []

for film in unique_films:
    # Use scorer=fuzz.token_set_ratio to handle titles like "Substance, The"
    match, score = process.extractOne(film, online_index.keys(), scorer=fuzz.token_set_ratio)
    
    # --- SAFETY FILTERS ---
    # 1. If the match is too long compared to the original film, it's a false positive
    length_ratio = len(match) / len(film) if len(film) > 0 else 0
    
    # 2. If the film title is very short (e.g. "Her"), the score must be nearly perfect
    min_score = 95 if len(film) < 10 else 85
    
    # 3. Validate whether the match is acceptable
    is_valid = score >= min_score and length_ratio < 2.5
    
    if is_valid:
        final_mapping.append({
            "oscar_title": film,
            "imsdb_title": match,
            "script_url": online_index[match],
            "match_score": score,
            "status": "Check"
        })
    else:
        final_mapping.append({
            "oscar_title": film,
            "imsdb_title": None,
            "script_url": None,
            "match_score": score,
            "status": "No Match"
        })

# Create DataFrame and display
df_mapping = pd.DataFrame(final_mapping)


In [ ]:
df_mapping


In [ ]:
# 1. Create a DataFrame with only records that have a URL (no NaNs)
df_matches = df_mapping.dropna(subset=['script_url']).copy()

# 2. Calculate statistics
total_original = len(df_mapping)
total_found = len(df_matches)
percentage = (total_found / total_original) * 100

print(f"--- MATCHING SUMMARY ---")
print(f"Total movies searched: {total_original}")
print(f"Scripts found: {total_found}")
print(f"Effectiveness: {percentage:.2f}%")

# 3. Show the first 10 matches for visual validation
print("\nFirst 10 matches:")
print(df_matches[['oscar_title', 'imsdb_title', 'match_score']].head(10))


## 5b. Matching Attempt 2 — Normalized Title Ratio

Uses `fuzz.ratio` on normalized titles (lowercased, articles stripped, special characters removed). More precise — avoids the subset-matching problem. Result: **16.67% match rate** with far fewer false positives.

> The low overall match rate is expected: most Oscar nominees predate the IMSDb collection, which skews toward 1980s–2010s films.

> **Note:** The kernel crashed after this cell due to the O(n²) nested loop over ~1,674 × ~1,300 titles. This is a known performance issue to address in the next iteration.

In [ ]:
import re

def normalize_title(t):
    # Convert to lowercase and remove leading "The", "A", "An" and special characters
    t = t.lower().strip()
    t = re.sub(r'^(the|a|an)\s+', '', t)
    return re.sub(r'[^a-z0-9 ]+', '', t)

final_mapping = []

for film in unique_films:
    # 1. Normalize the title being searched
    film_norm = normalize_title(film)
    
    # 2. Find the best match using plain ratio (stricter)
    # Compare the normalized film against normalized versions of the index
    best_score = 0
    best_match = None
    
    for imsdb_title in online_index.keys():
        imsdb_norm = normalize_title(imsdb_title)
        
        # Use plain ratio so "Big House" is not equal to "Big"
        current_score = fuzz.ratio(film_norm, imsdb_norm)
        
        if current_score > best_score:
            best_score = current_score
            best_match = imsdb_title
            
    # 3. Acceptance threshold (90 is very safe for normalized titles)
    if best_score >= 90:
        final_mapping.append({
            "oscar_title": film,
            "imsdb_title": best_match,
            "script_url": online_index[best_match],
            "score": best_score
        })
    else:
        final_mapping.append({
            "oscar_title": film, "imsdb_title": None, 
            "script_url": None, "score": best_score
        })

df_mapping = pd.DataFrame(final_mapping)


In [ ]:
df_mapping


In [ ]:
# 1. Create a DataFrame with only records that have a URL (no NaNs)
df_matches = df_mapping.dropna(subset=['script_url']).copy()

# 2. Calculate statistics
total_original = len(df_mapping)
total_found = len(df_matches)
percentage = (total_found / total_original) * 100

print(f"--- MATCHING SUMMARY ---")
print(f"Total movies searched: {total_original}")
print(f"Scripts found: {total_found}")
print(f"Effectiveness: {percentage:.2f}%")

# 3. Show the first 10 matches for visual validation
print("\nFirst 10 matches:")
print(df_matches[['oscar_title', 'imsdb_title', 'score']].head(10))
